## Learning Objectives

By the end of this tutorial, you will be able to

1. explain what the SPEAR algorithm estimates from time-ordered user-resource interactions,
2. prepare a small interaction table for SPEAR, and
3. run SPEAR to obtain ranked user expertise scores and resource quality scores.

## Target audience

This tutorial is aimed at social scientists who work with digital behavioral data such as bookmarks, likes, reposts, citations, hyperlinks, or platform activity logs. Basic Python and tabular data knowledge is sufficient.

## Setting up the computational environment

The repository contains all code and a small public example dataset. Binder installs the packages listed in `requirements.txt`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from spear import run_spear, sqrt_credit, constant_credit

plt.rcParams["figure.figsize"] = (7, 4)

## Duration

Around 20 minutes.

## Social Science Usecase(s)

SPEAR was introduced for distinguishing experts from spammers in folksonomies and social bookmarking systems (Noll et al., 2009). The same data structure also appears in many social science settings: people interact with pages, papers, posts, channels, hashtags, or other resources over time. SPEAR is useful when early interaction with later high-quality resources is substantively meaningful, for example when identifying knowledgeable curators, influential early adopters, or suspicious accounts that repeatedly promote low-quality resources.

## What SPEAR does

Following the original formulation by Noll et al. (2009) and the accompanying project documentation (Noll, n.d.), SPEAR takes a chronological list of activities `(timestamp, user, resource)` as input. It creates a weighted user-resource matrix in which users who found a resource earlier receive more credit when the same resource is later collected by others. It then iteratively estimates two mutually reinforcing quantities:

- **expertise**: users receive high scores when they are connected to high-quality resources, especially early;
- **quality**: resources receive high scores when they are connected to high-expertise users.

The algorithm is related to HITS-style authority/hub ranking (Kleinberg, 1999) but adds time-aware credit scoring. The implementation in this submission was checked against the public Julia example by Bleier (2013).

## Input data

The input file [`data/social_bookmarks.csv`](data/social_bookmarks.csv) is a toy social-bookmarking dataset. Each row is one observed activity.

Required columns:

- `timestamp`: when the activity happened;
- `user`: account or actor identifier;
- `resource`: item, URL, document, post, or other resource identifier.

The `tag` column is included to resemble bookmarking data, but the implementation below does not use it.

In [ ]:
activities = pd.read_csv("data/social_bookmarks.csv", parse_dates=["timestamp"])
activities.head()

In [ ]:
pd.DataFrame({
    "number_of_activities": [len(activities)],
    "number_of_users": [activities["user"].nunique()],
    "number_of_resources": [activities["resource"].nunique()],
    "first_activity": [activities["timestamp"].min()],
    "last_activity": [activities["timestamp"].max()],
})

## Run SPEAR

The default credit function is `sqrt_credit`, corresponding to `C(x)=sqrt(x)`: earlier users on a resource gain more credit as additional users adopt the same resource, but the square root dampens very large differences. The number of iterations controls how often expertise and quality update each other. The example converges quickly, but using 20 iterations is a safe default for small demonstrations.

In [ ]:
result = run_spear(activities, iterations=20, credit=sqrt_credit)

print("User expertise ranking")
display(result.expertise)

print("Resource quality ranking")
display(result.quality)

## Inspect the weighted adjacency matrix

Rows are users, columns are resources. Larger values indicate more credit for a user-resource relation after applying the time-aware credit function.

In [ ]:
result.adjacency.round(3)

## Visualize the output

The scores are normalized to sum to one, so they are best read as relative rankings within the analyzed dataset rather than as absolute measures.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

result.expertise.sort_values("expertise").plot.barh(x="user", y="expertise", ax=axes[0], legend=False, color="#377eb8")
axes[0].set_title("User expertise")
axes[0].set_xlabel("score")

result.quality.sort_values("quality").plot.barh(x="resource", y="quality", ax=axes[1], legend=False, color="#4daf4a")
axes[1].set_title("Resource quality")
axes[1].set_xlabel("score")

plt.tight_layout()

## Compare with a non-temporal variant

If all observed edges receive constant credit, SPEAR behaves more like a non-temporal HITS-style ranking. Comparing both settings helps assess how much of the result comes from early discovery rather than just from being connected to many resources.

In [ ]:
temporal = result.expertise.rename(columns={"expertise": "sqrt_credit"})
non_temporal = run_spear(activities, credit=constant_credit).expertise.rename(columns={"expertise": "constant_credit"})
comparison = temporal.merge(non_temporal, on="user")
comparison

## Interpreting and reporting SPEAR results

When using SPEAR in a study, report at least:

1. the definition of an activity, user, and resource;
2. the observation window and ordering variable;
3. the credit function and iteration settings;
4. preprocessing choices, especially duplicate handling and bot/spam filtering;
5. robustness checks, such as comparing `sqrt_credit` with `constant_credit`.

SPEAR can identify promising expert or quality candidates, but the scores should be validated against domain knowledge or external labels when they are used for substantive claims.

## Conclusion

You have loaded chronological user-resource data, run SPEAR, inspected the weighted adjacency matrix, and compared time-aware and non-temporal rankings. The same workflow can be applied to larger social bookmarking, platform interaction, citation, or hyperlink datasets after adapting the input columns.

## AI Use Acknowledgement

This submission was prepared with assistance from an AI coding assistant. The assistant helped draft explanatory text, create the example notebook structure, implement and test the Python code, and check Binder readiness. The author reviewed, edited, and takes responsibility for the final content, code, and citations.

## References

Bleier, A. (2013). *SpearAlgorithm.jl* [Computer software]. GitHub. Retrieved May 11, 2026, from https://github.com/arnim/SpearAlgorithm.jl

Kleinberg, J. M. (1999). Authoritative sources in a hyperlinked environment. *Journal of the ACM, 46*(5), 604–632. https://doi.org/10.1145/324133.324140

Noll, M. G. (n.d.). *SPEAR algorithm*. Retrieved May 11, 2026, from https://www.michael-noll.com/projects/spear-algorithm/

Noll, M. G., Au Yeung, C.-m., Gibbins, N., Meinel, C., & Shadbolt, N. (2009). Telling experts from spammers. In *Proceedings of the 32nd International ACM SIGIR Conference on Research and Development in Information Retrieval* (pp. 612–619). Association for Computing Machinery. https://doi.org/10.1145/1571941.1572046